# Tarea 2 – CSP: Asignación de Microservicios a Servidores

**Problema:** Asignar 8 microservicios (M1..M8) a 3 servidores físicos (S1, S2, S3).

**Restricciones:**
- **Capacidad:** Cada servidor aloja como máximo 3 microservicios.
- **Anti-afinidad:** Los pares (M1,M2), (M3,M4), (M5,M6), (M1,M5) no pueden compartir servidor.

**Función de peso:** $w(\text{asignación}) = \begin{cases} 1 & \text{si todas las restricciones se cumplen} \\ 0 & \text{en caso contrario} \end{cases}$

In [7]:
import sys, os

# Asegurar que el paquete src sea importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'src')))

import time
from csp.problem import (
    VARIABLES, DOMAIN, ANTI_AFFINITY_PAIRS, MAX_PER_SERVER,
    is_valid, total_violations, weight,
)
from csp.backtracking import backtracking_search
from csp.beam_search import beam_search
from csp.local_search import icm_search

print('Módulos cargados exitosamente.')

Módulos cargados exitosamente.


## 2.1 – Búsqueda Backtracking con Forward Checking

In [8]:
start = time.time()
bt_result = backtracking_search()
bt_time = time.time() - start

print(f"Resultado Backtracking + FC: {bt_result}")
print(f"Válido: {is_valid(bt_result) if bt_result else False}")
print(f"Peso: {weight(bt_result) if bt_result else 0}")
print(f"Tiempo: {bt_time:.6f} s")

Resultado Backtracking + FC: {'M1': 'S1', 'M2': 'S2', 'M5': 'S2', 'M6': 'S1', 'M3': 'S1', 'M4': 'S2', 'M7': 'S3', 'M8': 'S3'}
Válido: True
Peso: 1
Tiempo: 0.000089 s


## 2.2 – Beam Search (K configurable)

In [9]:
K_VALUES = [2, 3, 5, 10, 20]
beam_results = {}

for k in K_VALUES:
    start = time.time()
    result = beam_search(k=k)
    elapsed = time.time() - start
    valid = is_valid(result) if result else False
    beam_results[k] = {'result': result, 'valid': valid, 'time': elapsed}
    print(f"K={k:>3d} | Válido: {valid} | Tiempo: {elapsed:.6f}s | Asignación: {result}")

# Elegir el mejor k que encontró una solución válida
best_beam_k = None
for k in K_VALUES:
    if beam_results[k]['valid']:
        best_beam_k = k
        break

print(f"\nMenor K que encuentra una solución válida: {best_beam_k}")

K=  2 | Válido: True | Tiempo: 0.000108s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
K=  3 | Válido: True | Tiempo: 0.000077s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
K=  5 | Válido: True | Tiempo: 0.000097s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
K= 10 | Válido: True | Tiempo: 0.000467s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
K= 20 | Válido: True | Tiempo: 0.000317s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S1', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}

Menor K que encuentra una solución válida: 2


## 2.3 – Búsqueda Local (ICM)

In [10]:
# Ejecutar ICM múltiples veces con diferentes semillas para evaluar robustez
NUM_RUNS = 10
icm_results = []

for seed in range(NUM_RUNS):
    start = time.time()
    result = icm_search(max_iterations=100, seed=seed)
    elapsed = time.time() - start
    valid = is_valid(result) if result else False
    icm_results.append({'seed': seed, 'result': result, 'valid': valid, 'time': elapsed})
    print(f"Semilla {seed:>2d} | Válido: {valid} | Tiempo: {elapsed:.6f}s | Asignación: {result}")

success_rate = sum(1 for r in icm_results if r['valid']) / NUM_RUNS * 100
print(f"\nTasa de éxito ICM: {success_rate:.0f}% ({sum(1 for r in icm_results if r['valid'])}/{NUM_RUNS})")

Semilla  0 | Válido: False | Tiempo: 0.000083s | Asignación: None
Semilla  1 | Válido: False | Tiempo: 0.000075s | Asignación: None
Semilla  2 | Válido: True | Tiempo: 0.000034s | Asignación: {'M1': 'S2', 'M2': 'S1', 'M3': 'S1', 'M4': 'S2', 'M5': 'S1', 'M6': 'S3', 'M7': 'S3', 'M8': 'S2'}
Semilla  3 | Válido: True | Tiempo: 0.000051s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S3', 'M4': 'S1', 'M5': 'S2', 'M6': 'S3', 'M7': 'S2', 'M8': 'S3'}
Semilla  4 | Válido: True | Tiempo: 0.000033s | Asignación: {'M1': 'S3', 'M2': 'S2', 'M3': 'S1', 'M4': 'S3', 'M5': 'S2', 'M6': 'S3', 'M7': 'S1', 'M8': 'S1'}
Semilla  5 | Válido: True | Tiempo: 0.000033s | Asignación: {'M1': 'S1', 'M2': 'S2', 'M3': 'S3', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S3', 'M8': 'S3'}
Semilla  6 | Válido: True | Tiempo: 0.000031s | Asignación: {'M1': 'S3', 'M2': 'S2', 'M3': 'S3', 'M4': 'S2', 'M5': 'S2', 'M6': 'S1', 'M7': 'S1', 'M8': 'S3'}
Semilla  7 | Válido: True | Tiempo: 0.000030s | Asignación: {'M1': 'S2', 'M2': 'S3'

## 2.4 – Benchmarking y Comparación

In [11]:
import numpy as np

NUM_BENCHMARK_RUNS = 50

def benchmark(fn, label, runs=NUM_BENCHMARK_RUNS):
    """Ejecuta *fn* `runs` veces y recopila estadísticas de tiempo y validez."""
    times = []
    valid_count = 0
    for _ in range(runs):
        t0 = time.time()
        result = fn()
        t1 = time.time()
        times.append(t1 - t0)
        if result is not None and is_valid(result):
            valid_count += 1
    return {
        'label': label,
        'mean_time': np.mean(times),
        'std_time': np.std(times),
        'min_time': np.min(times),
        'max_time': np.max(times),
        'success_rate': valid_count / runs * 100,
    }

stats = [
    benchmark(backtracking_search, 'Backtracking+FC'),
    benchmark(lambda: beam_search(k=10), 'Beam Search (K=10)'),
    benchmark(lambda: icm_search(max_iterations=100), 'ICM Local Search'),
]

print(f"{'Algoritmo':<25s} {'Media (s)':>12s} {'Desv (s)':>12s} {'Mín (s)':>12s} {'Máx (s)':>12s} {'Éxito %':>10s}")
print('-' * 83)
for s in stats:
    print(f"{s['label']:<25s} {s['mean_time']:>12.6f} {s['std_time']:>12.6f} {s['min_time']:>12.6f} {s['max_time']:>12.6f} {s['success_rate']:>9.1f}%")

Algoritmo                    Media (s)     Desv (s)      Mín (s)      Máx (s)    Éxito %
-----------------------------------------------------------------------------------
Backtracking+FC               0.000034     0.000004     0.000031     0.000050     100.0%
Beam Search (K=10)            0.000197     0.000047     0.000157     0.000280     100.0%
ICM Local Search              0.000043     0.000017     0.000006     0.000084      88.0%


In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

labels = [s['label'] for s in stats]
means  = [s['mean_time'] * 1000 for s in stats]  # ms
stds   = [s['std_time']  * 1000 for s in stats]
rates  = [s['success_rate'] for s in stats]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Comparación de tiempos
axes[0].bar(labels, means, yerr=stds, capsize=5, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[0].set_ylabel('Tiempo medio (ms)')
axes[0].set_title('Comparación de Tiempo de Ejecución')
axes[0].tick_params(axis='x', rotation=15)

# Tasa de éxito
axes[1].bar(labels, rates, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[1].set_ylabel('Tasa de éxito (%)')
axes[1].set_title('Tasa de Validez de Soluciones')
axes[1].set_ylim(0, 110)
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(rates):
    axes[1].text(i, v + 2, f'{v:.0f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../evaluation/benchmark.png', dpi=150)
plt.show()
print('Gráfica guardada en evaluation/benchmark.png')

Gráfica guardada en evaluation/benchmark.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_10400\1620192021.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Análisis

**Backtracking + Forward Checking** es un método de búsqueda completo: *garantiza* encontrar una
solución válida si existe alguna. Forward Checking poda el árbol de búsqueda de forma temprana
eliminando valores del dominio que violan restricciones inmediatamente, combinado con la heurística
MRV (Minimum Remaining Values) para elegir primero la variable más restringida. Para este problema
de 8 variables y 3 valores es extremadamente rápido y siempre tiene éxito.

**Beam Search** es un método heurístico incompleto. Expande solo las K mejores asignaciones
parciales en cada nivel, usando el total de violaciones de restricciones como función de
puntuación. Con un K pequeño el haz puede colapsar y perder soluciones válidas; con un K
suficientemente grande (≥5–10 para este problema) encuentra soluciones de forma confiable
y se ejecuta muy rápido porque el factor de ramificación es solo 3.

**ICM (Iterated Conditional Modes)** comienza con una asignación completa aleatoria y de forma
voraz reasigna cada variable al valor que minimiza las violaciones. Converge rápido (a menudo
en 1–2 barridos) pero puede quedar atrapado en mínimos locales, por lo que su tasa de éxito
depende del estado aleatorio inicial. Ejecutarlo con múltiples semillas aleatorias (reinicios)
mitiga este problema.

**Conclusión:** Para un CSP pequeño como este, Backtracking+FC es la mejor opción: es exacto,
rápido y garantiza encontrar una solución. Beam Search ofrece un buen equilibrio cuando el
problema escala y los métodos exactos se vuelven demasiado costosos. ICM es el más rápido por
ejecución pero requiere reinicios para lograr alta confiabilidad, siendo más adecuado para
problemas muy grandes donde incluso Beam Search es demasiado costoso.